# Импортируем библиотеки

In [1]:
import requests
import pandas as pd
import time


In [5]:
count_text_lang = 35

In [4]:
headers = {
    "User-Agent": "LanguageDatasetParser/1.0 (student project; contact@example.com)"
}


In [6]:
LANGUAGES = {
    "English":    "en",
    "French":     "fr",
    "Spanish":    "es",
    "Portugeese": "pt",
    "Italian":    "it",
    "Russian":    "ru",
    "Sweedish":   "sv",
    "Malayalam":  "ml",
    "Dutch":      "nl",
    "Arabic":     "ar",
    "Turkish":    "tr",
    "German":     "de",
    "Tamil":      "ta",
    "Danish":     "da",
    "Kannada":    "kn",
    "Greek":      "el",
    "Hindi":      "hi",
}


Получает список случайных названий статей с Wikipedia

In [ ]:
def get_random_article_titles(lang_code, count):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
 
    params = {
        "action":      "query",
        "list":        "random",       
        "rnnamespace": 0,              
        "rnlimit":     count,          
        "format":      "json",
    }
 
    response = requests.get(url, params=params, headers=HEADERS, timeout=10)
    data = response.json()
    print(response.status_code, response.text[:200])
    titles = [item["title"] for item in data["query"]["random"]]
    return titles


Получает текст статьи Wikipedia по её названию.

In [ ]:
def get_article_text(lang_code, title):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
 
    params = {
        "action":      "query",
        "prop":        "extracts",     
        "exintro":     True,           
        "explaintext": True,           
        "titles":      title,
        "format":      "json",
    }
 
    response = requests.get(url, params=params, headers=headers, timeout=10)
    data = response.json()
 
    pages = data["query"]["pages"]
    page = next(iter(pages.values()))  
 
    text = page.get("extract", "")
    return text.strip()


  Разбивает текст на предложения по точке.
    Убирает слишком короткие предложения (меньше 30 символов).

In [ ]:
def split_into_sentences(text):

    sentences = text.split(".")
 
    result = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) > 30:  
            result.append(sentence)
 
    return result


Собирает тексты для одного языка.

In [ ]:
def parse_language(language_name, lang_code, texts_count):
    print(f"\n[{language_name}] Начинаем сбор текстов...")
 
    collected_texts = []
 
    try:
        titles = get_random_article_titles(lang_code, texts_count)
    except Exception as e:
        print(f"[{language_name}] Ошибка при получении списка статей: {e}")
        return []
    for i, title in enumerate(titles):
        try:
            text = get_article_text(lang_code, title)
 
            if not text:
                print(f"  [{i+1}/{len(titles)}] '{title}' — пустая статья, пропускаем")
                continue
 
            sentences = split_into_sentences(text)
            for sentence in sentences:
                collected_texts.append({
                    "Text":     sentence,
                    "Language": language_name
                })
 
            print(f"  [{i+1}/{len(titles)}] '{title}' — получено {len(sentences)} предложений")
 
            time.sleep(0.3)
 
        except Exception as e:
            print(f"  [{i+1}/{len(titles)}] '{title}' — ошибка: {e}")
            continue
 
    print(f"[{language_name}] Итого собрано: {len(collected_texts)} текстов")
    return collected_texts


In [ ]:
file  = 'parsed.csv'
def main():
    print("=" * 50)
    print("Парсер Wikipedia — сбор текстов для датасета")
    print("=" * 50)
 
    all_texts = []  
 
    for language_name, lang_code in LANGUAGES.items():
        texts = parse_language(language_name, lang_code, count_text_lang)
        all_texts.extend(texts)
 
        df_temp = pd.DataFrame(all_texts)
        df_temp.to_csv(file, index=False, encoding="utf-8")
        print(f"  Промежуточное сохранение: {len(all_texts)} текстов в {file}")
 
    df = pd.DataFrame(all_texts)
 
    print("\n" + "=" * 50)
    print("Сбор завершён!")
    print(f"Всего текстов: {len(df)}")
    print("\nРаспределение по языкам:")
    print(df["Language"].value_counts().to_string())
    print("=" * 50)
 
    # Сохраняем финальный файл
    df.to_csv(file, index=False, encoding="utf-8")
    print(f"\nФайл сохранён: {file}")


In [21]:
main()

Парсер Wikipedia — сбор текстов для датасета

[English] Начинаем сбор текстов...


TypeError: 'NoneType' object is not iterable